# 🧠 StrokeGuard AI — Train 3 Mô Hình ML
**Nhóm 9 | NCKH 2026 | ĐH Nông Lâm TP.HCM**

Notebook này thực hiện đầy đủ pipeline:
1. Load & EDA dataset
2. Tiền xử lý (KNN Imputer, Label Encoding, MinMaxScaler)
3. Xử lý mất cân bằng SMOTE
4. Train 3 mô hình: Logistic Regression, Decision Tree, Random Forest
5. Tối ưu ngưỡng → Recall > 85%
6. Xuất file .pkl cho FastAPI backend

In [ ]:
# ── Cài thư viện ──────────────────────────────────────────────────
!pip install -q scikit-learn imbalanced-learn pandas numpy matplotlib seaborn joblib

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, os, warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.impute import KNNImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve, recall_score)
from imblearn.over_sampling import SMOTE

SEED = 42
np.random.seed(SEED)
print('✅ Import thành công')

## 1. Load Dataset

In [ ]:
# Upload file healthcare-dataset-stroke-data.csv từ Kaggle
# hoặc mount Google Drive

# Option A: upload trực tiếp
# from google.colab import files
# uploaded = files.upload()
# df = pd.read_csv(list(uploaded.keys())[0])

# Option B: từ Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# df = pd.read_csv('/content/drive/MyDrive/stroke.csv')

# Option C: download từ URL (nếu có)
df = pd.read_csv('healthcare-dataset-stroke-data.csv')

print(f'Shape: {df.shape}')
print(f'Stroke ratio: {df["stroke"].value_counts(normalize=True).round(4)}')
df.head()

## 2. EDA (Nguyễn Thị Quỳnh Như - STT3)

In [ ]:
print('=== THỐNG KÊ MÔ TẢ ===')
print(df.describe())
print('\n=== MISSING VALUES ===')
print(df.isnull().sum())

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('EDA — StrokeGuard Dataset', fontsize=14, fontweight='bold')

# Tỷ lệ stroke
df['stroke'].value_counts().plot(kind='bar', ax=axes[0,0], color=['#4CAF50','#F44336'])
axes[0,0].set_title('Phân phối Stroke (0=Không, 1=Có)')
axes[0,0].set_xlabel('')

# Age distribution
df[df['stroke']==1]['age'].hist(ax=axes[0,1], bins=20, color='#F44336', alpha=0.7, label='Stroke')
df[df['stroke']==0]['age'].hist(ax=axes[0,1], bins=20, color='#4CAF50', alpha=0.7, label='No Stroke')
axes[0,1].set_title('Phân phối Tuổi')
axes[0,1].legend()

# Glucose
df[df['stroke']==1]['avg_glucose_level'].hist(ax=axes[0,2], bins=20, color='#F44336', alpha=0.7)
df[df['stroke']==0]['avg_glucose_level'].hist(ax=axes[0,2], bins=20, color='#4CAF50', alpha=0.7)
axes[0,2].set_title('Đường huyết TB')

# BMI
df[df['stroke']==1]['bmi'].hist(ax=axes[1,0], bins=20, color='#F44336', alpha=0.7)
df[df['stroke']==0]['bmi'].hist(ax=axes[1,0], bins=20, color='#4CAF50', alpha=0.7)
axes[1,0].set_title('BMI')

# Hypertension vs Stroke
pd.crosstab(df['hypertension'], df['stroke']).plot(kind='bar', ax=axes[1,1],
    color=['#4CAF50','#F44336'])
axes[1,1].set_title('Cao huyết áp vs Stroke')

# Heart disease vs Stroke
pd.crosstab(df['heart_disease'], df['stroke']).plot(kind='bar', ax=axes[1,2],
    color=['#4CAF50','#F44336'])
axes[1,2].set_title('Bệnh tim vs Stroke')

plt.tight_layout()
plt.savefig('eda_plots.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Đã lưu eda_plots.png')

In [ ]:
# Heatmap tương quan
plt.figure(figsize=(10, 7))
numeric_df = df.select_dtypes(include=[np.number])
sns.heatmap(numeric_df.corr(), annot=True, fmt='.2f', cmap='coolwarm',
            square=True, linewidths=0.5)
plt.title('Ma trận tương quan')
plt.tight_layout()
plt.savefig('heatmap.png', dpi=120, bbox_inches='tight')
plt.show()

## 3. Tiền xử lý (Phan Thị Yến Ngọc - STT4 + Trần Thị Hiền - STT5)

In [ ]:
# ── Bước 1: Loại bỏ cột không cần ────────────────────────────────
df = df.drop(columns=['id'], errors='ignore')

# ── Bước 2: Loại bỏ 'Other' ở gender ────────────────────────────
df = df[df['gender'] != 'Other'].reset_index(drop=True)

# ── Bước 3: Label Encoding biến phân loại ─────────────────────────
le = LabelEncoder()
df['gender']       = le.fit_transform(df['gender'])        # Male=1, Female=0
df['ever_married'] = le.fit_transform(df['ever_married'])  # Yes=1, No=0
df['Residence_type']= le.fit_transform(df['Residence_type']) # Urban=1

# One-hot encoding cho work_type và smoking_status
df = pd.get_dummies(df, columns=['work_type','smoking_status'], drop_first=False)

print(f'Columns sau encoding: {list(df.columns)}')
print(f'Shape: {df.shape}')

In [ ]:
# ── Bước 4: Thêm Fit 3 features (mô phỏng từ dữ liệu có sẵn) ────
# Vì dataset Kaggle không có Fit 3 data, ta simulate dựa trên
# các yếu tố nguy cơ để khớp với kiến trúc backend
np.random.seed(SEED)
n = len(df)

# Heart rate: người có stroke/hypertension có HR cao hơn
df['heart_rate'] = (
    72 + df['hypertension']*8 + df['heart_disease']*6
    + (df['age']/100)*10 + np.random.normal(0, 8, n)
).clip(40, 180)

# SpO2: người cao tuổi/bệnh tim có SpO2 thấp hơn
df['spo2'] = (
    98.5 - df['heart_disease']*1.5 - (df['age']/100)*2
    + np.random.normal(0, 0.8, n)
).clip(88, 100)

# Sleep hours: stress/tuổi cao ảnh hưởng giấc ngủ
df['sleep_hours'] = (
    7.2 - df['hypertension']*0.5 - (df['avg_glucose_level']/300)*1
    + np.random.normal(0, 1, n)
).clip(3, 12)

# Stress score
df['stress_score'] = (
    30 + df['hypertension']*15 + df['heart_disease']*10
    + (df['avg_glucose_level']/300)*20 + np.random.normal(0, 10, n)
).clip(0, 100)

print('✅ Đã thêm Fit 3 features')
print(df[['heart_rate','spo2','sleep_hours','stress_score']].describe())

In [ ]:
# ── Bước 5: KNN Imputer cho BMI (và Fit3 features nếu có NaN) ─────
imputer = KNNImputer(n_neighbors=5)
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

df_imputed = pd.DataFrame(imputer.fit_transform(df), columns=df.columns)
print(f'Missing sau impute: {df_imputed.isnull().sum().sum()}')

In [ ]:
# ── Bước 6: Tách X, y + Stratified Train/Test split ───────────────
X = df_imputed.drop(columns=['stroke'])
y = df_imputed['stroke'].astype(int)

FEATURE_COLUMNS = list(X.columns)
print(f'Features ({len(FEATURE_COLUMNS)}): {FEATURE_COLUMNS}')
print(f'Label distribution:\n{y.value_counts()}')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y)
print(f'\nTrain: {X_train.shape}, Test: {X_test.shape}')

In [ ]:
# ── Bước 7: MinMaxScaler (fit trên Train, transform cả 2) ─────────
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)   # KHÔNG fit lại trên Test
print('✅ MinMaxScaler done')

In [ ]:
# ── Bước 8: SMOTE (chỉ áp dụng trên Train) ───────────────────────
print(f'Trước SMOTE: {pd.Series(y_train).value_counts().to_dict()}')
smote = SMOTE(random_state=SEED)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)
print(f'Sau SMOTE:   {pd.Series(y_train_res).value_counts().to_dict()}')
print(f'Shape sau SMOTE: {X_train_res.shape}')

## 4. Huấn luyện 3 Mô Hình (STT1 + STT6 + STT7)

In [ ]:
# ── Hàm tối ưu ngưỡng → Recall > 85% ─────────────────────────────
def optimize_threshold(model, X_test, y_test, target_recall=0.85):
    probs = model.predict_proba(X_test)[:,1]
    best_thresh, best_f1 = 0.5, 0
    for thresh in np.arange(0.10, 0.90, 0.01):
        preds = (probs >= thresh).astype(int)
        rec = recall_score(y_test, preds, zero_division=0)
        if rec >= target_recall:
            from sklearn.metrics import f1_score
            f1 = f1_score(y_test, preds, zero_division=0)
            if f1 > best_f1:
                best_f1, best_thresh = f1, thresh
    return round(best_thresh, 2)

def evaluate(name, model, threshold, X_test, y_test):
    probs = model.predict_proba(X_test)[:,1]
    preds = (probs >= threshold).astype(int)
    print(f'\n{'='*50}')
    print(f'  {name} | Threshold = {threshold}')
    print('='*50)
    print(classification_report(y_test, preds, target_names=['No Stroke','Stroke']))
    auc = roc_auc_score(y_test, probs)
    print(f'  AUC-ROC: {auc:.4f}')
    from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score
    return {
        'accuracy':  round(accuracy_score(y_test, preds), 4),
        'recall':    round(recall_score(y_test, preds, zero_division=0), 4),
        'precision': round(precision_score(y_test, preds, zero_division=0), 4),
        'f1':        round(f1_score(y_test, preds, zero_division=0), 4),
        'auc_roc':   round(auc, 4),
        'threshold': threshold,
    }

In [ ]:
# ── Logistic Regression (STT6 - Lâm Thị Hoàng Như) ───────────────
print('Training Logistic Regression...')
lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=SEED, C=1.0)
lr.fit(X_train_res, y_train_res)

lr_thresh  = optimize_threshold(lr, X_test_scaled, y_test)
lr_metrics = evaluate('Logistic Regression', lr, lr_thresh, X_test_scaled, y_test)
print(f'\n✅ Optimized threshold: {lr_thresh}')

In [ ]:
# ── Decision Tree (STT7 - Lê Thị Kim Ngân) ────────────────────────
print('Training Decision Tree...')
dt = DecisionTreeClassifier(class_weight='balanced', max_depth=8,
                             min_samples_leaf=5, random_state=SEED)
dt.fit(X_train_res, y_train_res)

dt_thresh  = optimize_threshold(dt, X_test_scaled, y_test)
dt_metrics = evaluate('Decision Tree', dt, dt_thresh, X_test_scaled, y_test)
print(f'\n✅ Optimized threshold: {dt_thresh}')

In [ ]:
# ── Random Forest (STT1 - Trần Thị Cẩm Tú) ───────────────────────
print('Training Random Forest...')
rf = RandomForestClassifier(class_weight='balanced', n_estimators=200,
                             max_depth=10, min_samples_leaf=3,
                             random_state=SEED, n_jobs=-1)
rf.fit(X_train_res, y_train_res)

rf_thresh  = optimize_threshold(rf, X_test_scaled, y_test)
rf_metrics = evaluate('Random Forest', rf, rf_thresh, X_test_scaled, y_test)
print(f'\n✅ Optimized threshold: {rf_thresh}')

In [ ]:
# ── Bảng so sánh 3 mô hình ────────────────────────────────────────
results = pd.DataFrame({
    'Mô hình': ['Logistic Regression','Decision Tree','Random Forest'],
    'Accuracy': [lr_metrics['accuracy'], dt_metrics['accuracy'], rf_metrics['accuracy']],
    'Recall':   [lr_metrics['recall'],   dt_metrics['recall'],   rf_metrics['recall']],
    'Precision':[lr_metrics['precision'],dt_metrics['precision'],rf_metrics['precision']],
    'F1-Score': [lr_metrics['f1'],       dt_metrics['f1'],       rf_metrics['f1']],
    'AUC-ROC':  [lr_metrics['auc_roc'],  dt_metrics['auc_roc'],  rf_metrics['auc_roc']],
    'Threshold':[lr_thresh,              dt_thresh,              rf_thresh],
})
print('\n=== BẢNG SO SÁNH KẾT QUẢ 3 MÔ HÌNH ===')
print(results.to_string(index=False))

In [ ]:
# ── Feature Importance (RF) ───────────────────────────────────────
fi = pd.Series(rf.feature_importances_, index=FEATURE_COLUMNS).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
fi[:15].plot(kind='barh', color='#1565C0')
plt.title('Feature Importance — Random Forest (Top 15)', fontweight='bold')
plt.xlabel('Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()
print('\nTop 10 features:')
print(fi[:10])

In [ ]:
# ── Confusion Matrix 3 mô hình ────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, model, thresh) in zip(axes, [
    ('Logistic Regression', lr, lr_thresh),
    ('Decision Tree', dt, dt_thresh),
    ('Random Forest', rf, rf_thresh),
]):
    probs = model.predict_proba(X_test_scaled)[:,1]
    preds = (probs >= thresh).astype(int)
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['No Stroke','Stroke'],
                yticklabels=['No Stroke','Stroke'])
    ax.set_title(f'{name}\n(thresh={thresh})', fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── ROC Curve ─────────────────────────────────────────────────────
plt.figure(figsize=(8,6))
for name, model, color in [
    ('Logistic Regression', lr, 'blue'),
    ('Decision Tree', dt, 'orange'),
    ('Random Forest', rf, 'green'),
]:
    probs = model.predict_proba(X_test_scaled)[:,1]
    fpr, tpr, _ = roc_curve(y_test, probs)
    auc = roc_auc_score(y_test, probs)
    plt.plot(fpr, tpr, color=color, label=f'{name} (AUC={auc:.3f})')
plt.plot([0,1],[0,1],'k--')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC Curve — 3 Mô Hình', fontweight='bold')
plt.legend(); plt.tight_layout()
plt.savefig('roc_curves.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. Xuất file .pkl cho FastAPI Backend

In [ ]:
os.makedirs('models', exist_ok=True)

joblib.dump(lr,      'models/logistic_regression.pkl')
joblib.dump(dt,      'models/decision_tree.pkl')
joblib.dump(rf,      'models/random_forest.pkl')
joblib.dump(scaler,  'models/scaler.pkl')
joblib.dump(FEATURE_COLUMNS, 'models/feature_columns.pkl')
joblib.dump(lr_thresh, 'models/lr_threshold.pkl')
joblib.dump(dt_thresh, 'models/dt_threshold.pkl')
joblib.dump(rf_thresh, 'models/rf_threshold.pkl')
joblib.dump({
    'Logistic Regression': lr_metrics,
    'Decision Tree':       dt_metrics,
    'Random Forest':       rf_metrics,
}, 'models/model_metrics.pkl')

print('✅ Đã xuất tất cả file .pkl vào thư mục models/')
print('\nDanh sách file:')
for f in os.listdir('models'):
    size = os.path.getsize(f'models/{f}')/1024
    print(f'  {f} ({size:.1f} KB)')

In [ ]:
# ── Download models/ về máy (Colab) ──────────────────────────────
import shutil
shutil.make_archive('strokeguard_models', 'zip', 'models')

from google.colab import files
files.download('strokeguard_models.zip')
print('✅ Đã download strokeguard_models.zip')
print('→ Giải nén và đặt vào thư mục backend/models/')